# LightFM Hybrid Demo

This notebook demonstrates the `LightFMHybridModel` wrapper and shows a feature influence analysis.

In [ ]:
import pandas as pd

from src.evaluation import LightFMFeatureInfluenceAnalyzer
from src.evaluation import LightFMFeatureInfluenceConfig
from src.models.lightfm_model import LightFMHybridModel

In [ ]:
# Build a small ratings dataframe
ratings_dataframe = pd.DataFrame(
    {
        "userId": [1, 1, 1, 2, 2, 3, 3, 3, 4, 4],
        "movieId": [1, 2, 3, 1, 4, 2, 3, 5, 1, 5],
        "rating": [4.0, 3.5, 2.0, 4.5, 3.0, 4.0, 3.0, 5.0, 2.5, 4.5],
    }
)

# Build a small movies feature dataframe using engineered columns
movies_feature_dataframe = pd.DataFrame(
    {
        "movieId": [1, 2, 3, 4, 5, 6],
        "genre_Action": [1, 0, 0, 1, 0, 1],
        "genre_Comedy": [0, 1, 1, 0, 1, 0],
        "genre_tfidf_action": [0.9, 0.0, 0.0, 0.8, 0.0, 0.7],
        "genre_tfidf_comedy": [0.0, 0.8, 0.7, 0.0, 0.9, 0.0],
        "release_year": [1995, 1998, 2001, 2005, 2010, 2012],
    }
)

ratings_dataframe.head(), movies_feature_dataframe.head()

In [ ]:
# Fit LightFM hybrid model with item features
model = LightFMHybridModel(number_of_components=8, number_of_epochs=8, random_seed=7)
model.fit(ratings_dataframe=ratings_dataframe, movies_dataframe=movies_feature_dataframe)

# Predict one user-item score
predicted_score = model.predict_rating(user_identifier=1, movie_identifier=4)
print("Predicted score for user 1 on movie 4:", predicted_score)

# Top-3 recommendations for user 1 (unseen items only)
recommendations = model.recommend_top_n(user_identifier=1, number_of_recommendations=3)
print("Top-3 recommendations for user 1:")
for movie_id, score in recommendations:
    print(" ", movie_id, score)

## Feature Influence Analysis

Positive influence means removing the feature worsens the selected metric.

In [ ]:
# Create a tiny split for the demo analysis.
train_dataframe = ratings_dataframe.iloc[:8].copy()
validation_dataframe = ratings_dataframe.iloc[8:].copy()

influence_analyzer = LightFMFeatureInfluenceAnalyzer(
    LightFMFeatureInfluenceConfig(
        metric_name="rmse_value",
        number_of_components=8,
        number_of_epochs=6,
        random_seed=7,
    )
)

feature_influence_dataframe = influence_analyzer.analyze(
    train_dataframe=train_dataframe,
    validation_dataframe=validation_dataframe,
    movies_dataframe=movies_feature_dataframe,
)
feature_influence_dataframe

In [ ]:
# Show strongest positive and negative features for accuracy.
print("Most positive influence:")
print(feature_influence_dataframe.head(3)[["feature_name", "influence_score", "influence_direction"]])

print("\nMost negative influence:")
print(feature_influence_dataframe.tail(3)[["feature_name", "influence_score", "influence_direction"]])